# WitnessSim Analysis


In [ ]:
import json, re, glob, pickle
from pathlib import Path
from collections import defaultdict

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.ndimage import gaussian_filter1d
from scipy.stats import pearsonr, spearmanr
from sklearn.linear_model import Ridge
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from transformers import AutoTokenizer, AutoModelForCausalLM

import sys
sys.path.append(str(BASE_DIR))
from config import DEFAULT_MODEL, TARGET_LAYER, TOKEN_START, BATCH_SIZE, BASE_DIR

# ── Paths ──────────────────────────────────────────────────────────────────
DENOISED_VECTORS  = BASE_DIR / "data/emotion_vectors_denoised.pt"
REAL_TX_DIR       = BASE_DIR / "data"          # real depositions
SIM_OUTPUT_DIR    = BASE_DIR / "output"                    # synthetic output
CACHE_DIR         = BASE_DIR / "cache"
CACHE_DIR.mkdir(exist_ok=True)

# 6 Witness Sim dimensions (in consistent order)
SIM_DIMS = ["A", "C", "K", "P", "R", "V"]
SIM_DIM_LABELS = {
    "A": "Agreeableness",
    "C": "Composure",
    "K": "Knowledge",
    "P": "Pressure",
    "R": "Rigidity",
    "V": "Volatility",
}
N_BINS   = 20   # for arc interpolation (matches main notebook)
TOP_EMOS = ["calm", "anxious", "hostile", "defiant", "resigned", "suspicious", "confident", "warm"]

print("Imports OK")

## Setup


In [ ]:
vecs_dict = torch.load(DENOISED_VECTORS, map_location="cpu", weights_only=False)
emotions  = sorted(vecs_dict.keys())
mat       = torch.stack([vecs_dict[e] for e in emotions], dim=0).float()
mat       = F.normalize(mat, dim=1)
emotion_to_idx = {e: i for i, e in enumerate(emotions)}

print(f"{len(emotions)} emotion vectors: {emotions}")

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

tokenizer = AutoTokenizer.from_pretrained(DEFAULT_MODEL)
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    DEFAULT_MODEL, torch_dtype=torch.bfloat16, device_map="auto"
)
model.eval()
print("Model loaded.")

## Encoding Helpers


In [ ]:
@torch.inference_mode()
def encode_texts(texts, layer=TARGET_LAYER, token_start=TOKEN_START, batch_size=BATCH_SIZE):
    """Return (N, D) float32 mean-pooled hidden states from `layer`."""
    all_vecs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        enc   = tokenizer(batch, return_tensors="pt", padding=True,
                          truncation=True, max_length=512).to(device)
        out   = model(**enc, output_hidden_states=True)
        h     = out.hidden_states[layer]          # (B, T, D)
        h     = h[:, token_start:, :].mean(dim=1) # (B, D)
        all_vecs.append(h.float().cpu())
    return torch.cat(all_vecs, dim=0)

def project_to_emotions(vecs):
    """Cosine-project (N, D) → (N, 23) emotion scores."""
    vecs_n = F.normalize(vecs, dim=1)
    return (vecs_n @ mat.T).numpy()

def interpolate_to_bins(scores, n_bins=N_BINS):
    """Interpolate (T, E) → (n_bins, E)."""
    T, E   = scores.shape
    x_orig = np.linspace(0, 1, T)
    x_new  = np.linspace(0, 1, n_bins)
    out    = np.zeros((n_bins, E))
    for e in range(E):
        out[:, e] = np.interp(x_new, x_orig, scores[:, e])
    return out

print("Helpers defined.")

## Parse and Encode Synthetic Transcripts


In [ ]:
def parse_synthetic_transcript(tx_path):
    """
    Parse a Witness Sim transcript into turns.
    Format:
        [Turn N]
        Q: question text (possibly multi-line)
        A: answer text (possibly multi-line)
           Δ: C▼0.010   ← optional, stripped

    Returns list of dicts: {turn_index, q_text, a_text}
    Turn index is 1-based to match the deltas file.
    """
    text = Path(tx_path).read_text(encoding="utf-8", errors="replace")
    # Strip header block (everything up to and including the === separator)
    sep = "=" * 60
    if sep in text:
        text = text[text.index(sep) + len(sep):].strip()

    TURN_RE  = re.compile(r'^\[Turn (\d+)\]', re.MULTILINE)
    DELTA_RE = re.compile(r'^\s*Δ:.*$', re.MULTILINE)

    turns = []
    parts = TURN_RE.split(text)   # ['', '1', body1, '2', body2, ...]
    it = iter(parts[1:])
    for turn_num_str, body in zip(it, it):
        turn_idx = int(turn_num_str)
        body = DELTA_RE.sub("", body)

        q_m = re.search(r'Q:\s*(.+?)(?=\nA:|\Z)', body, re.DOTALL)
        a_m = re.search(r'A:\s*(.+?)(?=\n\[Turn|\Z)', body, re.DOTALL)
        if not q_m or not a_m:
            continue
        q = " ".join(q_m.group(1).split())
        a = " ".join(a_m.group(1).split())
        if a:
            turns.append({"turn_index": turn_idx, "q_text": q, "a_text": a})
    return turns


def load_delta_file(delta_path):
    """Return dict: turn_index (1-based) → delta record."""
    records = json.loads(Path(delta_path).read_text())
    return {r["turn"]: r for r in records}


def process_synthetic_pair(tx_path, delta_path):
    """
    Returns list of merged turn dicts:
      turn_index, q_text, a_text,
      state (6D), state_delta (6D), events, scores, pressure, question_type,
      emotion_scores (23D np array)
    """
    turns  = parse_synthetic_transcript(tx_path)
    deltas = load_delta_file(delta_path)
    if not turns:
        return []

    # Encode all witness answers in one batch
    a_texts = [t["a_text"] for t in turns]
    vecs    = encode_texts(a_texts)
    scores  = project_to_emotions(vecs)   # (T, 23)

    merged = []
    for i, turn in enumerate(turns):
        tidx   = turn["turn_index"]
        delta  = deltas.get(tidx, {})
        merged.append({
            **turn,
            "state":         delta.get("state", {}),
            "state_delta":   delta.get("state_delta", {}),
            "events":        delta.get("events", []),
            "sim_scores":    delta.get("scores", {}),
            "pressure":      delta.get("pressure", None),
            "question_type": delta.get("question_type", None),
            "emotion_scores": scores[i],   # (23,)
        })
    return merged


print("Parse helpers defined.")


In [ ]:
SIM_CACHE = CACHE_DIR / "synthetic_turns.pkl"

if SIM_CACHE.exists():
    print("Loading synthetic turns from cache...")
    with open(SIM_CACHE, "rb") as f:
        synthetic_data = pickle.load(f)
    print(f"  {len(synthetic_data)} (witness, style) pairs loaded.")
else:
    synthetic_data = {}   # key: (witness, style)

    tx_files = sorted(SIM_OUTPUT_DIR.glob("*/*_transcript.txt"))
    print(f"Found {len(tx_files)} synthetic transcripts to process.")

    for tx_path in tx_files:
        style    = tx_path.stem.replace("_transcript", "")
        witness  = tx_path.parent.name
        key      = (witness, style)
        delta_path = tx_path.parent / f"{style}_deltas.json"
        if not delta_path.exists():
            print(f"  SKIP {key}: no delta file")
            continue

        print(f"  Processing {witness}/{style}...", end=" ", flush=True)
        turns = process_synthetic_pair(tx_path, delta_path)
        synthetic_data[key] = turns
        print(f"{len(turns)} turns")

    with open(SIM_CACHE, "wb") as f:
        pickle.dump(synthetic_data, f)
    print(f"\nCached {len(synthetic_data)} pairs → {SIM_CACHE}")

## Load Real Deposition Scores


In [ ]:
DEPO_RESULTS_DIR = BASE_DIR / "data/depo_results"   # written by encode_depositions.ipynb

real_data = {}   # key: transcript stem → {emotion_scores: (T, 23)}

npz_files = sorted(DEPO_RESULTS_DIR.glob("*.npz"))
print(f"Found {len(npz_files)} cached deposition files in {DEPO_RESULTS_DIR}")

for npz_path in npz_files:
    data = np.load(npz_path, allow_pickle=True)
    witn_scores = data["witn_scores"]   # (T_witness, 23)
    if witn_scores.shape[0] < 3:
        continue
    real_data[npz_path.stem] = {"emotion_scores": witn_scores.astype(np.float32)}
    print(f"  [loaded] {npz_path.stem}: {witn_scores.shape[0]} witness turns")

print(f"\nLoaded {len(real_data)} real depositions (witness turns only).")


## Arc Similarity Analysis


In [ ]:
# ── Build interpolated arc arrays ─────────────────────────────────────────

def arcs_from_real(real_data, n_bins=N_BINS):
    """Return (N_transcripts, n_bins, 23) array."""
    arcs = []
    for stem, d in real_data.items():
        sc = d["emotion_scores"]   # (T, 23)
        sc = sc[~np.isnan(sc).any(axis=1)]   # drop NaN turns
        if sc.shape[0] < 3:
            continue
        arcs.append(interpolate_to_bins(sc, n_bins))
    return np.stack(arcs, axis=0)  # (N, n_bins, 23)


def arcs_from_synthetic(synthetic_data, style=None, n_bins=N_BINS):
    """
    Return (N, n_bins, 23) array.
    If style is given, restrict to that style; else use all styles.
    """
    arcs = []
    for (witness, sty), turns in synthetic_data.items():
        if style and sty != style:
            continue
        sc = np.stack([t["emotion_scores"] for t in turns])  # (T, 23)
        sc = sc[~np.isnan(sc).any(axis=1)]   # drop NaN turns
        if sc.shape[0] < 3:
            continue
        arcs.append(interpolate_to_bins(sc, n_bins))
    return np.stack(arcs, axis=0) if arcs else np.empty((0, n_bins, 23))


real_arcs = arcs_from_real(real_data)         # (N_real, N_BINS, 23)
syn_arcs  = arcs_from_synthetic(synthetic_data)  # (N_syn, N_BINS, 23)

print(f"Real arcs:      {real_arcs.shape}")
print(f"Synthetic arcs: {syn_arcs.shape}")


In [ ]:
syn_neutral = arcs_from_synthetic(synthetic_data, style="neutral")

real_mean   = real_arcs.mean(axis=0)          # (N_BINS, 23)
syn_all_mean = syn_arcs.mean(axis=0)          # (N_BINS, 23)
syn_neu_mean = syn_neutral.mean(axis=0) if syn_neutral.shape[0] > 0 else None

# Find top-8 emotions by average real score
top_idxs    = real_mean.mean(axis=0).argsort()[::-1][:8]
top_emotions = [emotions[i] for i in top_idxs]

x = np.linspace(0, 100, N_BINS)
fig, axes = plt.subplots(1, 2, figsize=(18, 5), sharey=False)

for ax, mean_arc, title in [
    (axes[0], real_mean,    f"Real depositions (n={real_arcs.shape[0]})"),
    (axes[1], syn_all_mean, f"Synthetic all styles (n={syn_arcs.shape[0]})"),
]:
    for i, emo in zip(top_idxs, top_emotions):
        y = gaussian_filter1d(mean_arc[:, i], sigma=1.5)
        ax.plot(x, y, label=emo, lw=2)
    ax.set_title(title, fontsize=13)
    ax.set_xlabel("Deposition position (%)")
    ax.set_ylabel("Mean emotion score")
    ax.legend(fontsize=8, ncol=2)
    ax.grid(alpha=0.3)

plt.suptitle("Mean Witness Emotion Arcs", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(CACHE_DIR / "q1_mean_arcs.png", dpi=150)
plt.show()

In [ ]:
# ── Witness-to-source-doc mapping (from attorney_questions Source: headers) ─
WITNESS_TO_DOCS = {
    "catherine_jackson":   ["MNKOI0004123186", "MNKOI0004569338"],
    "hugh_oneill":         ["MNKOI0004568397"],
    "jane_williams":       ["MNKOI0005218871"],
    "jeffrey_kilper":      ["MNKOI0004560796"],
    "john_adams":          ["MNKOI0004556728"],
    "kirk_dumont":         ["MNKOI0004561730"],
    "mark_pugh":           ["MNKOI0004570602"],
    "michael_wessler":     ["MNKOI0004557138"],
    "tiffany_kilper":      ["MNKOI0004560979"],
    "todd_dean":           ["MNKOI0004569936"],
}

# Top emotions for arc line plots (by mean real score)
all_real_scores = np.vstack([d["emotion_scores"] for d in real_data.values()])
TOP_EMO_IDXS    = all_real_scores.mean(axis=0).argsort()[::-1][:6]
TOP_EMOS        = [emotions[i] for i in TOP_EMO_IDXS]

def safe_pearsonr(a, b):
    """Pearson r — returns 0 if either array contains NaN, is constant, or r is NaN."""
    if np.any(np.isnan(a)) or np.any(np.isnan(b)):
        return 0.0
    if a.std() < 1e-9 or b.std() < 1e-9:
        return 0.0
    r, _ = pearsonr(a, b)
    return 0.0 if np.isnan(r) else float(r)

def per_emotion_similarity(real_arc, sim_arc):
    """Pearson r per emotion between two (N_BINS, 23) arcs. Returns (23,)."""
    return np.array([safe_pearsonr(real_arc[:, i], sim_arc[:, i])
                     for i in range(real_arc.shape[1])])

def mean_similarity(real_arc, sim_arc):
    return float(per_emotion_similarity(real_arc, sim_arc).mean())

print(f"Top 6 emotions for arc plots: {TOP_EMOS}")


In [ ]:
# ── Set to None to run all witnesses; set to a name to test one ────────────
TEST_WITNESS = None

# ── Per-witness comparison: arc plots + per-emotion similarity heatmap ─────
x_pos       = np.linspace(0, 100, N_BINS)
line_colors = plt.cm.tab10(np.linspace(0, 1, len(TOP_EMOS)))

best_match_summary = []

witness_iter = (
    [(TEST_WITNESS, WITNESS_TO_DOCS[TEST_WITNESS])]
    if TEST_WITNESS else sorted(WITNESS_TO_DOCS.items())
)

for witness, doc_ids in witness_iter:

    # ── real arc ──────────────────────────────────────────────────────────
    real_arcs_w = []
    for doc_id in doc_ids:
        if doc_id in real_data:
            sc = real_data[doc_id]["emotion_scores"]
            sc = sc[~np.isnan(sc).any(axis=1)]   # drop NaN turns
            if sc.shape[0] >= 3:
                real_arcs_w.append(interpolate_to_bins(sc, N_BINS))
    if not real_arcs_w:
        print(f"  SKIP {witness}: no real depo cache ({doc_ids})")
        continue
    real_arc_w = np.mean(real_arcs_w, axis=0)   # (N_BINS, 23)

    # ── all simulated arcs ────────────────────────────────────────────────
    sim_arcs = {}
    for (w, sty), turns in synthetic_data.items():
        if w != witness or len(turns) < 3:
            continue
        sc = np.stack([t["emotion_scores"] for t in turns])
        sc = sc[~np.isnan(sc).any(axis=1)]   # drop NaN turns
        if len(sc) < 3:
            continue
        sim_arcs[sty] = interpolate_to_bins(sc, N_BINS)
    if not sim_arcs:
        print(f"  SKIP {witness}: no simulated styles")
        continue

    # ── per-emotion similarity matrix: (23, n_styles) ─────────────────────
    style_order   = sorted(sim_arcs.keys())
    sim_matrix    = np.stack(
        [per_emotion_similarity(real_arc_w, sim_arcs[s]) for s in style_order],
        axis=1,
    )   # (23, n_styles)
    mean_rs       = sim_matrix.mean(axis=0)
    sorted_idxs   = mean_rs.argsort()[::-1]
    ranked_styles = [style_order[i] for i in sorted_idxs]
    ranked_matrix = sim_matrix[:, sorted_idxs]

    best_style = ranked_styles[0]
    best_r     = float(mean_rs[sorted_idxs[0]])
    best_match_summary.append({
        "witness":       witness,
        "best_style":    best_style,
        "similarity_r":  best_r,
        "n_styles":      len(ranked_styles),
        "ranking":       list(zip(ranked_styles, mean_rs[sorted_idxs].round(3).tolist())),
        "per_emotion_r": {s: sim_matrix[:, j].tolist()
                          for j, s in enumerate(style_order)},
    })

    # ── Fig A: arc line plots sorted best→worst ────────────────────────────
    ncols = 1 + len(ranked_styles)
    fig, axes = plt.subplots(1, ncols, figsize=(3.2 * ncols, 3.8), sharey=False)

    # Real panel
    ax = axes[0]
    for emo, c in zip(TOP_EMOS, line_colors):
        i = emotion_to_idx[emo]
        ax.plot(x_pos, gaussian_filter1d(real_arc_w[:, i], sigma=0.8, mode='nearest'),
                color=c, lw=2, label=emo)
    ax.set_xlim(0, 100)
    ax.set_title(f"Real\n({len(real_arcs_w)} depo{'s' if len(real_arcs_w)>1 else ''})",
                 fontsize=9, fontweight="bold")
    ax.set_ylabel("Emotion score"); ax.set_xlabel("Pos (%)"); ax.grid(alpha=0.3)
    ax.legend(fontsize=6, loc="upper right")

    # Sim panels sorted best→worst
    for col_idx, sty in enumerate(ranked_styles, start=1):
        ax  = axes[col_idx]
        arc = sim_arcs[sty]
        r   = float(mean_rs[sorted_idxs[col_idx - 1]])
        is_best = (col_idx == 1)
        for emo, c in zip(TOP_EMOS, line_colors):
            i = emotion_to_idx[emo]
            ax.plot(x_pos, gaussian_filter1d(arc[:, i], sigma=0.8, mode='nearest'),
                    color=c, lw=2)
        ax.set_xlim(0, 100)
        ax.set_title(f"{'★ ' if is_best else ''}{sty}\nr={r:.3f}",
                     fontsize=9,
                     color="darkgreen" if is_best else ("gray" if col_idx == len(ranked_styles) else "black"),
                     fontweight="bold" if is_best else "normal")
        ax.set_xlabel("Pos (%)"); ax.grid(alpha=0.3)
        if is_best:
            for sp in ax.spines.values():
                sp.set_edgecolor("darkgreen"); sp.set_linewidth(2.5)

    fig.suptitle(f"{witness.replace('_',' ').title()}  —  best: {best_style} (mean r={best_r:.3f})",
                 fontsize=11, fontweight="bold")
    plt.tight_layout()
    plt.savefig(CACHE_DIR / f"q1_arcs_{witness}.png", dpi=150, bbox_inches="tight")
    plt.show()

    # ── Fig B: per-emotion similarity heatmap ─────────────────────────────
    emo_variance = ranked_matrix.var(axis=1)
    emo_order    = emo_variance.argsort()[::-1]
    plot_matrix  = ranked_matrix[emo_order, :]
    emo_labels   = [emotions[i] for i in emo_order]

    fig2, ax2 = plt.subplots(figsize=(max(6, 0.7 * len(ranked_styles)), 8))
    im = ax2.imshow(plot_matrix, aspect="auto", cmap="RdYlGn", vmin=-1, vmax=1)
    ax2.set_xticks(range(len(ranked_styles)))
    ax2.set_xticklabels(
        [f"{'★' if s == best_style else ''}{s}" for s in ranked_styles],
        rotation=40, ha="right", fontsize=9,
    )
    ax2.set_yticks(range(len(emotions)))
    ax2.set_yticklabels(emo_labels, fontsize=8)
    plt.colorbar(im, ax=ax2, label="Pearson r (real vs sim arc)")
    for ei in range(len(emotions)):
        for si in range(len(ranked_styles)):
            v = plot_matrix[ei, si]
            if abs(v) > 0.3:
                ax2.text(si, ei, f"{v:.2f}", ha="center", va="center",
                         fontsize=6, color="white" if abs(v) > 0.6 else "black")
    ax2.axvline(0, color="darkgreen", lw=3, alpha=0.6)
    ax2.set_title(
        f"{witness.replace('_',' ').title()} — per-emotion arc similarity\n"
        f"(cols sorted best→worst mean r; rows sorted by cross-style variance)",
        fontsize=10, fontweight="bold",
    )
    plt.tight_layout()
    plt.savefig(CACHE_DIR / f"q1_heatmap_{witness}.png", dpi=150, bbox_inches="tight")
    plt.show()

print(f"\nDone. Processed {len(best_match_summary)} witness(es).")
print("Set TEST_WITNESS = None to run all.")


In [ ]:
print(f"{'Witness':<22} {'Best style':<16} {'r':>6}  {'Full ranking (style: r)'}")
print("-" * 90)
for row in sorted(best_match_summary, key=lambda r: -r["similarity_r"]):
    ranking_str = "  ".join(f"{s}:{r:.3f}" for s, r in row["ranking"])
    print(f"{row['witness']:<22} {row['best_style']:<16} {row['similarity_r']:>6.3f}  {ranking_str}")

from collections import Counter
style_wins = Counter(row["best_style"] for row in best_match_summary)
print(f"\nBest-match style frequency across all witnesses:")
for sty, count in style_wins.most_common():
    print(f"  {sty:<16} {count}")


In [ ]:
# ── Plot 1b: correlation heatmap — real mean arc vs. each synthetic style ──

styles = sorted({sty for (_, sty) in synthetic_data.keys()})
corr_matrix = np.zeros((len(emotions), len(styles)))   # corr per emotion per style

for j, style in enumerate(styles):
    sty_arcs = arcs_from_synthetic(synthetic_data, style=style)
    if sty_arcs.shape[0] == 0:
        continue
    sty_mean = sty_arcs.mean(axis=0)   # (N_BINS, 23)
    for i in range(len(emotions)):
        corr_matrix[i, j] = safe_pearsonr(real_mean[:, i], sty_mean[:, i])

fig, ax = plt.subplots(figsize=(14, 9))
im = ax.imshow(corr_matrix, aspect="auto", cmap="RdYlGn", vmin=-1, vmax=1)
ax.set_xticks(range(len(styles))); ax.set_xticklabels(styles, rotation=45, ha="right")
ax.set_yticks(range(len(emotions))); ax.set_yticklabels(emotions)
plt.colorbar(im, ax=ax, label="Pearson r (real mean vs. style mean arc)")
ax.set_title("Q1 — Arc correlation: real depositions vs. each synthetic style",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(CACHE_DIR / "q1_corr_heatmap.png", dpi=150)
plt.show()

print("Mean correlation across all emotions and styles:",
      corr_matrix.mean().round(3))


In [ ]:
# ── Plot 1c: PCA embedding — real vs. synthetic arcs ──────────────────────
# Flatten each arc to (N_BINS * 23,), then PCA-2D

real_flat = real_arcs.reshape(real_arcs.shape[0], -1)   # (N_real, N_BINS*23)

# Build syn_flat and style_list together with the same NaN-filter as arcs_from_synthetic
syn_flat_rows = []
style_list    = []
for (witness, sty), turns in synthetic_data.items():
    sc = np.stack([t["emotion_scores"] for t in turns])
    sc = sc[~np.isnan(sc).any(axis=1)]   # drop NaN turns
    if sc.shape[0] < 3:
        continue
    syn_flat_rows.append(interpolate_to_bins(sc, N_BINS).flatten())
    style_list.append(sty)

syn_flat = np.array(syn_flat_rows) if syn_flat_rows else np.empty((0, N_BINS * 23))

combined    = np.vstack([real_flat, syn_flat])
combined    = np.nan_to_num(combined, nan=0.0)   # safety net

scaler      = StandardScaler()
combined_sc = scaler.fit_transform(combined)

pca = PCA(n_components=2, random_state=42)
xy  = pca.fit_transform(combined_sc)

nr = real_flat.shape[0]

style_colors = {s: c for s, c in zip(styles,
                plt.cm.tab20(np.linspace(0, 1, len(styles))))}

fig, ax = plt.subplots(figsize=(12, 8))
ax.scatter(xy[:nr, 0], xy[:nr, 1], c="black", alpha=0.5, s=30, label="Real", zorder=3)

seen_styles = set()
for i, sty in enumerate(style_list):
    label = sty if sty not in seen_styles else "_nolegend_"
    seen_styles.add(sty)
    pt = xy[nr + i]
    ax.scatter(pt[0], pt[1], c=[style_colors.get(sty, "gray")],
               alpha=0.7, s=40, label=label)

ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
ax.set_title("Q1 — PCA of flattened emotion arcs (real vs. synthetic)",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=7, ncol=3, loc="upper right")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(CACHE_DIR / "q1_pca.png", dpi=150)
plt.show()


## Event Detection


In [ ]:
# ── Event-aligned averaging ────────────────────────────────────────────────
WINDOW = 5   # turns before and after event

# Collect all turns as a flat list with their sequence position
by_series = {}   # (witness, style) → list of turns sorted by turn_index
for (witness, style), turns in synthetic_data.items():
    by_series[(witness, style)] = sorted(turns, key=lambda t: t["turn_index"])

event_windows = defaultdict(list)   # event_label → list of (2W+1, 23) arrays

for (witness, style), turns in by_series.items():
    T   = len(turns)
    arr = np.stack([t["emotion_scores"] for t in turns])  # (T, 23)
    # Replace NaN scores with per-emotion column mean so they don't pollute windows
    col_means = np.nanmean(arr, axis=0)
    arr = np.where(np.isnan(arr), col_means[np.newaxis, :], arr)

    for i, turn in enumerate(turns):
        for ev in turn["events"]:
            label = ev["label"]
            lo = max(0, i - WINDOW)
            hi = min(T, i + WINDOW + 1)
            pad_lo = WINDOW - (i - lo)
            pad_hi = WINDOW - (hi - i - 1)
            window_arr = arr[lo:hi]   # (≤2W+1, 23)
            # Pad with NaN if near edge
            window_arr = np.pad(
                window_arr,
                ((pad_lo, pad_hi), (0, 0)),
                constant_values=np.nan,
            )   # (2W+1, 23)
            event_windows[label].append(window_arr)

print("Event counts:")
for label, wins in event_windows.items():
    print(f"  {label}: {len(wins)} occurrences")


In [ ]:
# ── Plot event-aligned emotion signatures ─────────────────────────────────
# For each event type, show the mean emotion trajectory around the event.

event_labels = sorted(event_windows.keys())
x_win = np.arange(-WINDOW, WINDOW + 1)

# Also compute baseline (non-event turns) mean per emotion
non_event_scores = []
for (witness, style), turns in by_series.items():
    for t in turns:
        if not t["events"]:
            non_event_scores.append(t["emotion_scores"])
baseline = np.nanmean(np.array(non_event_scores), axis=0)  # (23,) — nanmean ignores bad encodings

for label in event_labels:
    wins   = np.array(event_windows[label])       # (N, 2W+1, 23)
    mean_w = np.nanmean(wins, axis=0)             # (2W+1, 23)
    delta  = mean_w - baseline[np.newaxis, :]     # deviation from non-event baseline

    # Top 6 emotions by max absolute deviation in ±2 turns of event
    center_window = delta[WINDOW-2:WINDOW+3]
    top_idx = np.abs(center_window).max(axis=0).argsort()[::-1][:6]
    top_emo = [emotions[i] for i in top_idx]

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # Left: raw scores
    ax = axes[0]
    for i, emo in zip(top_idx, top_emo):
        y = gaussian_filter1d(mean_w[:, i], sigma=0.8)
        ax.plot(x_win, y, label=emo, lw=2)
    ax.axvline(0, color="red", lw=2, ls="--", label="Event")
    ax.set_title(f"{label} — Raw emotion scores", fontsize=11)
    ax.set_xlabel("Turns relative to event")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

    # Right: deviation from baseline
    ax = axes[1]
    for i, emo in zip(top_idx, top_emo):
        y = gaussian_filter1d(delta[:, i], sigma=0.8)
        ax.plot(x_win, y, label=emo, lw=2)
    ax.axvline(0, color="red", lw=2, ls="--", label="Event")
    ax.axhline(0, color="gray", lw=0.8)
    ax.set_title(f"{label} — Δ vs. non-event baseline", fontsize=11)
    ax.set_xlabel("Turns relative to event")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

    plt.suptitle(f"Q3 — Event signature: {label}  (n={wins.shape[0]})",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    safe = label.replace(" ", "_").lower()
    plt.savefig(CACHE_DIR / f"q3_event_{safe}.png", dpi=150)
    plt.show()


In [ ]:
# ── Per-witness WITNESS COMBATIVE event-aligned plot ─────────────────────
# Shows raw emotion scores in a ±5 turn window around each WITNESS COMBATIVE
# event, averaged per witness. Only witnesses whose archetype triggers the
# event (A_t < 0.28 AND R_t > 0.60) will appear.

from collections import defaultdict

KEEP_EMOTIONS = ['calm', 'defiant', 'hostile', 'angry', 'proud', 'anxious']
keep_idx = [emotions.index(e) for e in KEEP_EMOTIONS]

_WINDOW    = 5
_x_win     = np.arange(-_WINDOW, _WINDOW + 1)
_event_lbl = "WITNESS COMBATIVE"

_witness_windows = defaultdict(list)
for (witness, style), turns in by_series.items():
    T   = len(turns)
    arr = np.stack([t["emotion_scores"] for t in turns])
    col_means = np.nanmean(arr, axis=0)
    arr = np.where(np.isnan(arr), col_means[np.newaxis, :], arr)

    for i, turn in enumerate(turns):
        for ev in turn["events"]:
            if ev["label"] != _event_lbl:
                continue
            lo     = max(0, i - _WINDOW)
            hi     = min(T, i + _WINDOW + 1)
            pad_lo = _WINDOW - (i - lo)
            pad_hi = _WINDOW - (hi - i - 1)
            window_arr = np.pad(arr[lo:hi], ((pad_lo, pad_hi), (0, 0)), constant_values=np.nan)
            _witness_windows[witness].append(window_arr)

_witnesses = sorted(_witness_windows.keys())
_n_wit     = len(_witnesses)
print(f"Witnesses with COMBATIVE events: {_n_wit} — {_witnesses}")
print({w: len(v) for w, v in _witness_windows.items()})

_ncols = 2
_nrows = int(np.ceil(_n_wit / _ncols))
fig, axes = plt.subplots(_nrows, _ncols, figsize=(14, 4 * _nrows))
axes = axes.flatten()

_colors = plt.cm.tab10.colors

for _row, _witness in enumerate(_witnesses):
    _windows  = np.stack(_witness_windows[_witness])   # (n_events, 11, 23)
    _mean_raw = np.nanmean(_windows, axis=0)           # (11, 23)
    _n_events = _windows.shape[0]

    for k, (idx, emo) in enumerate(zip(keep_idx, KEEP_EMOTIONS)):
        axes[_row].plot(_x_win, _mean_raw[:, idx], label=emo, color=_colors[k])

    axes[_row].axvline(0, color="red", linestyle="--", linewidth=1.5)
    axes[_row].set_title(f"{_witness}  (n={_n_events} events)")
    axes[_row].set_xlabel("Turns relative to event")
    axes[_row].set_ylabel("Emotion score")
    axes[_row].legend(fontsize=8)
    axes[_row].grid(True, alpha=0.3)

for ax in axes[_n_wit:]:
    ax.set_visible(False)

fig.suptitle("WITNESS COMBATIVE — per witness, raw emotion scores", fontsize=13)
plt.tight_layout()
plt.savefig("combative_per_witness_raw.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
X_rows, Y_rows, meta_rows = [], [], []

for (witness, style), turns in synthetic_data.items():
    for t in turns:
        if not t.get("state"):
            continue
        X_rows.append(t["emotion_scores"])
        Y_rows.append([t["state"][d] for d in SIM_DIMS])
        meta_rows.append({"witness": witness, "style": style, "t": t["turn_index"]})

X = np.array(X_rows)   # (N_turns, 23)
Y = np.array(Y_rows)   # (N_turns, 6)
print(f"Design matrix: {X.shape[0]} turns, {X.shape[1]} emotions, {Y.shape[1]} state dims")


In [ ]:
# Fit a linear map Y→X (6D state predicts emotion scores)
# Residuals = emotion_scores - predicted_emotion_scores

from sklearn.linear_model import Ridge

Ys_sc = StandardScaler().fit_transform(Y)   # 6D state, standardized
Xs_sc = StandardScaler().fit_transform(X)   # 23D emotion, standardized

# Predict each emotion from the 6D state
X_pred = np.zeros_like(Xs_sc)
for i in range(len(emotions)):
    reg = Ridge(alpha=1.0).fit(Ys_sc, Xs_sc[:, i])
    X_pred[:, i] = reg.predict(Ys_sc)

residuals = Xs_sc - X_pred   # (N, 23)  — variance unexplained by 6D state

# R² for each emotion
r2_per_emotion = 1 - (residuals**2).mean(axis=0) / (Xs_sc**2).mean(axis=0)
print("R² (emotion ~ 6D state) per emotion:")
for i, emo in enumerate(emotions):
    print(f"  {emo}: {r2_per_emotion[i]:.3f}")

print(f"\nMean R²: {r2_per_emotion.mean():.3f}")
print(f"Emotions largely unexplained (R² < 0.1):")
for i, emo in enumerate(emotions):
    if r2_per_emotion[i] < 0.1:
        print(f"  {emo}")

In [ ]:
pca_res = PCA(n_components=6, random_state=42)
res_pcs = pca_res.fit_transform(residuals)   # (N, 6)

print("Residual PCA explained variance ratio:")
for k, v in enumerate(pca_res.explained_variance_ratio_):
    print(f"  PC{k+1}: {v*100:.1f}%")

# Top emotion loadings per residual PC
print("\nTop emotion loadings on each residual PC:")
for k in range(4):
    comp = pca_res.components_[k]   # (23,)
    top_pos = [(emotions[i], comp[i]) for i in comp.argsort()[::-1][:4]]
    top_neg = [(emotions[i], comp[i]) for i in comp.argsort()[:4]]
    print(f"  PC{k+1} (EVR={pca_res.explained_variance_ratio_[k]*100:.1f}%)")
    print(f"    +: {top_pos}")
    print(f"    -: {top_neg}")

# Plot residual PC loadings as bar chart
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for k, ax in enumerate(axes.flat):
    comp   = pca_res.components_[k]
    order  = np.argsort(comp)[::-1]
    colors = ["steelblue" if v > 0 else "salmon" for v in comp[order]]
    ax.barh([emotions[i] for i in order[::-1]], comp[order[::-1]], color=colors[::-1])
    ax.axvline(0, color="black", lw=0.8)
    ax.set_title(f"Residual PC{k+1} ({pca_res.explained_variance_ratio_[k]*100:.1f}%)",
                 fontsize=11, fontweight="bold")
    ax.set_xlabel("Loading")
    ax.grid(alpha=0.3)

plt.suptitle("PCA of emotion residuals (variance not explained by 6D state)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(CACHE_DIR / "q3_residual_pca.png", dpi=150)
plt.show()

In [ ]:
# Can emotion scores (or residuals) predict event occurrence?
# Binary classification: event turn vs. non-event turn.

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import StratifiedKFold

# Build binary labels for each event type
meta_arr = np.array([
    [any(ev["label"] == lbl for ev in [
        ev for t in by_series.get((m["witness"], m["style"]), [])
        for ev in t["events"]
        if t["turn_index"] == m["turn"]
    ]) for lbl in event_labels]
    for m in meta_rows
], dtype=float)   # (N, n_events)

print("Event base rates:")
for j, lbl in enumerate(event_labels):
    print(f"  {lbl}: {meta_arr[:, j].mean()*100:.1f}%")

print("\nEvent detection AUC (5-fold CV):")
print(f"  {'Event':<25} {'Emotion AUC':>12} {'Residual AUC':>13}")

for j, lbl in enumerate(event_labels):
    y = meta_arr[:, j].astype(int)
    if y.sum() < 10:
        print(f"  {lbl:<25} too few positives")
        continue

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    aucs_emo, aucs_res = [], []
    for train_idx, test_idx in skf.split(Xs_sc, y):
        for feat, auc_list in [(Xs_sc, aucs_emo), (residuals, aucs_res)]:
            clf = LogisticRegression(max_iter=500, C=1.0, class_weight="balanced")
            clf.fit(feat[train_idx], y[train_idx])
            prob = clf.predict_proba(feat[test_idx])[:, 1]
            auc_list.append(roc_auc_score(y[test_idx], prob))

    print(f"  {lbl:<25} {np.mean(aucs_emo):>12.3f} {np.mean(aucs_res):>13.3f}")

print("\n(Residual AUC > Emotion AUC → event signature goes beyond 6D state prediction)")

In [ ]:
# Do specific residual PCs spike at event turns?

fig, axes = plt.subplots(len(event_labels), 4, figsize=(18, 4 * len(event_labels)),
                         sharey="col")
if len(event_labels) == 1:
    axes = axes[np.newaxis, :]

for j, lbl in enumerate(event_labels):
    y = meta_arr[:, j].astype(bool)
    for k in range(4):
        ax   = axes[j, k]
        ev_  = res_pcs[y,  k]
        non_ = res_pcs[~y, k]
        ax.hist(non_, bins=40, alpha=0.6, density=True, label="non-event", color="steelblue")
        ax.hist(ev_,  bins=40, alpha=0.7, density=True, label="event",     color="salmon")
        ax.set_title(f"{lbl}\nResidual PC{k+1}", fontsize=8)
        ax.legend(fontsize=7)
        ax.grid(alpha=0.3)

plt.suptitle("Residual PC distributions: event vs. non-event turns",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(CACHE_DIR / "q3_residual_pc_distributions.png", dpi=150)
plt.show()

## Summary


In [ ]:
print("=" * 70)
print("SUMMARY")
print("=" * 70)

print("
Arc Similarity")
print(f"  Real transcripts:      {real_arcs.shape[0]}")
print(f"  Synthetic transcripts: {syn_arcs.shape[0]} ({len(styles)} styles)")
print(f"  Mean arc correlation (real vs. synthetic): {corr_matrix.mean():.3f}")
print(f"  Best-correlated style: {styles[corr_matrix.mean(axis=0).argmax()]} "
      f"(mean r={corr_matrix.mean(axis=0).max():.3f})")

print("
Event Detection")
print(f"  Events detected: {list(event_windows.keys())}")
print(f"  Residual PCA EVR: " +
      ", ".join(f"PC{k+1}={v*100:.1f}%" for k, v in
               enumerate(pca_res.explained_variance_ratio_[:4])))
print("=" * 70)
